# 01 — Dataset preprocessing: from SecFaaS2Fog placement to priced BIM* instances

This notebook documents and executes the **transformation pipeline** that turns the original
SecFaaS2Fog placement dataset into the priced BIM* corpus used by the campaign.

```
original_dataset/                    experimentation.icsoc.bimstar          out/bimstar-priced/
├── applications.json   ──────────►  generator (seeded, deterministic) ──►  instances/{app}/{seed}/*.bimstar.json
└── infrastructures/{seed}/*.json                                          reports/*.csv (full provenance traces)
```

The generator (package `experimentation/icsoc/bimstar/`) performs, per application × infrastructure:

1. **Tasks & composition** — orchestration functions become BIM tasks; the orchestration grammar
   (`seq`/`par`/`if`) becomes a structured SEQ/AND/XOR tree (IF guards → XOR with p = 0.5 per branch).
2. **Candidates** — one candidate per (function, hosting option): original edge/fog/cloud nodes that
   satisfy the function's software/hardware requirements and can bind its required services, plus
   **generated cloud FaaS candidates** for every provider × region × memory size ≥ the requirement.
3. **Pricing (iPricings)** — each candidate gets an expected **monthly USD cost** computed from the
   machine-readable pricings in `pricings/` (AWS Lambda, Azure Functions, Google Cloud Functions),
   using simple **on-demand / consumption plans** driven by the decision variables:
   invocations/month, average duration, allocated memory, vCPUs and region (§2 below).
4. **Latency model** — the original node-to-node latencies are kept; missing pairs (cloud pools)
   are generated with a **provider/geography-aware model** (§3 below).
5. **Security** — information-flow labels are propagated over the orchestration (SecFaaS2Fog typing)
   with **per-variable dataflow** (`security.output_propagation: variable`): trigger labels ride on
   the named variables, an output named like an existing variable keeps that variable's label, and
   only genuinely new variables inherit the task's level. Per-task minimum security thresholds
   therefore vary (e.g. arOrch: fLogin 1.0 but fCrop/fGeo 0.66), so the security objective term
   genuinely discriminates. Node capabilities map to scores
   (none = 0.33, pubKeyE = 0.66, pubKeyE+antiTamp = 1.0).
6. **Pricing constraints & canonical bounds** — budgets are anchored to a **certified feasible
   witness** (a cheapest-first backtracking assignment over AC-3-filtered pools that satisfies
   every transition bound and pool capacity): per-task budgets = max(p75 of eligible costs,
   witness cost); global budget = witness cost + `global_budget_slack` × (fold of local budgets −
   witness cost). Every instance is satisfiable by construction, with configurable difficulty,
   plus min–max normalization bounds shared by every engine (§4 below).

Everything is **deterministic**: the generator seed (12345) plus the dataset seed (146588263)
fully determine the corpus.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'docker-compose.yml').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'experimentation/icsoc'))

from experimentation.icsoc.bimstar.config import load_config
from experimentation.icsoc.bimstar.generator import generated_latency
from experimentation.icsoc.bimstar.pricing import FaaSPricing

import campaign
import analysis

DATASET = REPO_ROOT / 'experimentation/icsoc/original_dataset'
PRICINGS = REPO_ROOT / 'pricings'
CONFIG_PATH = REPO_ROOT / 'experimentation/icsoc/bimstar/configs/default.yml'
CORPUS_OUT = REPO_ROOT / 'experimentation/icsoc/out/bimstar-priced'
FIGURES = REPO_ROOT / 'experimentation/icsoc/out/results/figures'

config = load_config(CONFIG_PATH)
print('generator config loaded from', CONFIG_PATH.name)

## 1. Input dataset

Three FaaS applications (AR, Media Processing, Stock Market) and 20 infrastructure seeds ×
35 sizes (50–220 nodes, step 5). The campaign uses **one dataset seed (146588263)**.

In [ ]:
apps = json.loads((DATASET / 'applications.json').read_text())
app_rows = [{
    'application': a['orchestration_id'],
    'name': a['name'],
    'functions': len(a['functions']),
    'service_requirements': sum(len(f.get('service_reqs') or []) for f in a['functions']),
} for a in apps]
display(pd.DataFrame(app_rows))

seed_dirs = sorted(p.name for p in (DATASET / 'infrastructures').iterdir() if p.is_dir())
sizes = sorted({int(p.stem.split('_')[1]) for p in (DATASET / 'infrastructures' / seed_dirs[0]).glob('infrastructure_*.json')})
print(f'{len(seed_dirs)} dataset seeds; sizes {sizes[0]}..{sizes[-1]} step {sizes[1]-sizes[0]} ({len(sizes)} sizes)')

infra = json.loads((DATASET / 'infrastructures/146588263/infrastructure_50.json').read_text())
kinds = pd.Series([n['type'] for n in infra['nodes']]).value_counts()
print('infrastructure_50 (seed 146588263):', dict(kinds), '|', len(infra['links']), 'links |', len(infra['services']), 'services')

## 2. Pricing: on-demand iPricing usage (audited)

Cloud candidate costs come from the **iPricing variables files** in `pricings/` (refreshed from the
provider APIs), always using the plain on-demand / consumption plans — committed-use and savings-plan
fields present in the files are deliberately ignored:

| Provider | Monthly cost formula (on-demand) |
|---|---|
| AWS Lambda | `invocations × lambdaRequest + GB-s × x86.firstGbSeconds` |
| Azure Functions (consumption) | `invocations × execution + GB-s × gbSecond` |
| Google Cloud Functions | `invocations × request + vCPU-s × vcpuSecond + GiB-s × gibSecond` |

with `GB-s = invocations × duration_s × memory_GB`. At the campaign workload
(1M invocations/month × 120 ms) the AWS compute volume is ~0.5M GB-s ≪ the 6B GB-s tier-1 limit,
so the tier-1 rate is exact. **Decision variables** driving each candidate: memory size
(128–4096 MB, ≥ the function requirement), invocations/month, average duration, vCPUs and region.

The worked example below recomputes a cost by hand from the raw iPricing rates and checks it
matches `FaaSPricing.estimate` (the same checks run in the unit tests).

In [ ]:
pricing = FaaSPricing(PRICINGS)
workload = config['workload_defaults']
inv, dur = workload['invocations_per_month'], workload['avg_duration_ms']

rates = pricing.aws['pricesPerRegionAndArchitecture']['eu-west-1']
gbs = inv * (dur / 1000) * (1024 / 1024)
by_hand = inv * float(rates['lambdaRequest']) + gbs * float(rates['x86']['firstGbSeconds'])
estimated = pricing.estimate('aws', 'eu-west-1', invocations_per_month=inv, avg_duration_ms=dur, memory_mb=1024)
print(f'AWS eu-west-1, {inv:,.0f} inv x {dur} ms x 1024 MB:')
print(f'  requests  = {inv * float(rates["lambdaRequest"]):.4f} USD')
print(f'  compute   = {gbs:,.0f} GB-s x {float(rates["x86"]["firstGbSeconds"]):.10f} = {gbs * float(rates["x86"]["firstGbSeconds"]):.4f} USD')
print(f'  total     = {by_hand:.4f} USD/month  (estimate() = {estimated:.4f})')
assert abs(by_hand - estimated) < 1e-12

# Cost vs memory per provider/region under the campaign workload
memories = config['cloud_memory_mb_domain']
fig, ax = plt.subplots(figsize=(7.2, 3.6))
for provider, regions in config['cloud_regions'].items():
    for region in regions:
        costs = [pricing.estimate(provider, region, invocations_per_month=inv,
                                  avg_duration_ms=dur, memory_mb=m) for m in memories]
        ax.plot(memories, costs, marker='o', ms=3,
                label=f'{provider} {region}', alpha=0.85)
ax.set(xlabel='allocated memory (MB)', ylabel='USD / month', xscale='log',
       title=f'On-demand FaaS cost — {inv:,.0f} invocations x {dur} ms')
ax.set_xticks(memories, labels=[str(m) for m in memories])
ax.legend(fontsize=7, ncols=2)
fig.tight_layout()
analysis.save_fig(fig, 'preprocessing_cloud_cost_vs_memory', FIGURES)

## 3. Latency model: provider/region-aware realism

Original node-to-node latencies come from the dataset. Pairs involving generated cloud pools use a
**provider + geography model** with *disjoint* ranges per class, so by construction:

- links between regions of the **same provider** (private backbone) are faster than links between
  **different providers** (public peering) in the same geography class, and
- **intra-geography** links are faster than **inter-geography** ones.

| Class | Range (ms) |
|---|---|
| same provider, same region | 1–3 |
| same provider, intra-geo | 8–18 |
| cross provider, intra-geo | 20–35 |
| same provider, inter-geo | 60–85 |
| cross provider, inter-geo | 90–130 |
| original node → EU cloud / US cloud | 20–60 / 80–150 |

In [ ]:
cloud_pools = [(f'{p}.faas.{r}', p, r)
               for p, regions in config['cloud_regions'].items() for r in regions]
pool_kind = {pid: 'CLOUD_FAAS' for pid, _, _ in cloud_pools}
pool_meta = {pid: {'commercial_provider': p, 'region': r} for pid, p, r in cloud_pools}

n = len(cloud_pools)
matrix = np.zeros((n, n))
for i, (a, _, _) in enumerate(cloud_pools):
    for j, (b, _, _) in enumerate(cloud_pools):
        matrix[i, j] = 0 if i == j else generated_latency(
            a, b, pool_kind, pool_meta, config, 12345, '146588263')

fig, ax = plt.subplots(figsize=(6.4, 5.2))
im = ax.imshow(matrix, cmap='viridis')
labels = [pid.replace('.faas.', '\n') for pid, _, _ in cloud_pools]
ax.set_xticks(range(n), labels=labels, fontsize=7, rotation=45, ha='right')
ax.set_yticks(range(n), labels=labels, fontsize=7)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{matrix[i, j]:.0f}', ha='center', va='center',
                color='white' if matrix[i, j] > 60 else 'black', fontsize=7)
ax.set_title('Generated cloud-to-cloud latency (ms)')
fig.colorbar(im, shrink=0.8)
fig.tight_layout()
analysis.save_fig(fig, 'preprocessing_cloud_latency_matrix', FIGURES)

# Structural realism check (also enforced by unit tests)
lat_cfg = config['latency_generation']
assert lat_cfg['cloud_same_provider_intra_geo_ms'][1] < lat_cfg['cloud_cross_provider_intra_geo_ms'][0]
assert lat_cfg['cloud_same_provider_inter_geo_ms'][1] < lat_cfg['cloud_cross_provider_inter_geo_ms'][0]
print('realism property holds: same provider < cross provider, intra-geo < inter-geo')

## 4. Corpus generation (deterministic)

Generates the 105 priced instances (3 apps × 35 sizes × seed 146588263) with budget constraints and
canonical normalization bounds, then validates every instance against the BIM* JSON Schema.
Eligible candidate sets come from **latency-arc-consistent (AC-3) pool domains** filtered by the
per-task security thresholds. Since AC-3 only guarantees arc consistency, budgets are anchored to a
**certified feasible witness** (backtracking over the filtered domains honouring transitions and
capacities), which keeps every instance satisfiable by construction. Skipped when the corpus
already exists.

In [ ]:
if not any((CORPUS_OUT / 'instances').glob('*/*/*.bimstar.json')):
    subprocess.run([
        sys.executable, '-m', 'experimentation.icsoc.bimstar.cli', 'generate',
        '--dataset', str(DATASET), '--pricing-dir', str(PRICINGS),
        '--config', str(CONFIG_PATH), '--seed', '12345',
        '--out', str(CORPUS_OUT), '--dataset-seeds', '146588263',
    ], cwd=REPO_ROOT, check=True)

result = subprocess.run([
    sys.executable, '-m', 'experimentation.icsoc.bimstar.cli', 'validate',
    '--instances', str(CORPUS_OUT / 'instances'),
    '--schema', str(REPO_ROOT / 'schemas/general/bimstar.schema.json'),
    '--report', str(CORPUS_OUT / 'reports/validation_report.json'),
], cwd=REPO_ROOT, capture_output=True, text=True)
print(result.stdout.strip() or result.stderr.strip())
assert result.returncode == 0

## 5. Budget derivation traces

`reports/budget_trace.csv` records, per instance and task: the security threshold, the number of
eligible candidates after AC-3 + security filtering, the certified witness cost, the per-task
budget (p75 of eligible costs, raised to the witness cost if needed) and, per instance, the global
budget (witness cost + `global_budget_slack` × headroom up to the local-budget fold) and the
canonical normalization bounds.

In [ ]:
budget = pd.read_csv(CORPUS_OUT / 'reports/budget_trace.csv')
task_rows = budget[budget.task != '__global__']
global_rows = budget[budget.task == '__global__'].copy()
global_rows['infra_size'] = global_rows.infrastructure.str.split('_').str[1].astype(int)

print(f"latency-feasible task rows: {task_rows.latency_feasible.mean():.1%} | "
      f"security-feasible: {task_rows.security_feasible.mean():.1%}")
display(task_rows.groupby('application')[['eligible_candidates', 'ac_pool_domain_size', 'local_budget']]
        .median().rename(columns=lambda c: f'median_{c}'))

fig, ax = plt.subplots(figsize=(6.8, 3.2))
for app, group in global_rows.groupby('application'):
    ax.plot(group.infra_size, group.global_budget, marker='.', label=app)
ax.set(xlabel='infrastructure size (nodes)', ylabel='global budget (USD/month)')
ax.legend()
fig.tight_layout()
analysis.save_fig(fig, 'preprocessing_global_budgets', FIGURES)

### Security thresholds per task (per-variable information flow)

With per-variable label propagation the thresholds vary across tasks, so security is a real
optimization trade-off instead of a saturated constraint (the classic non-interference join
creeps every task to `top` on pipeline-shaped orchestrations).


In [ ]:
thresholds = (task_rows.drop_duplicates(['application', 'task'])
              .pivot_table(index='application', columns='task',
                           values='security_threshold', aggfunc='first'))
display(thresholds)
assert (thresholds.nunique(axis=1) > 1).all(), 'thresholds should vary within every app'


## 6. Corpus characterization (derived instance attributes)

Quantifies the experimental variable domains: candidates, pools, constraints per kind and the
binding-space size per instance — the covariates used later in the evaluation.

In [ ]:
summary_csv = campaign.write_corpus_summary()
corpus_df = pd.read_csv(summary_csv)
display(corpus_df.groupby('application')[['n_tasks', 'n_candidates', 'n_pools',
        'n_constraints_local', 'n_constraints_global', 'n_transitions',
        'n_budget_constraints', 'log10_binding_space']].agg(['min', 'max']))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for app, group in corpus_df.groupby('application'):
    axes[0].plot(group['infra_size'], group['n_candidates'], marker='.', label=app)
    axes[1].plot(group['infra_size'], group['log10_binding_space'], marker='.', label=app)
axes[0].set(xlabel='infrastructure size (nodes)', ylabel='candidates')
axes[1].set(xlabel='infrastructure size (nodes)', ylabel='log10 |binding space|')
axes[0].legend()
fig.tight_layout()
analysis.save_fig(fig, 'preprocessing_corpus_characterization', FIGURES)
print(f'{len(corpus_df)} instances characterized -> {summary_csv}')

## Outputs

- `out/bimstar-priced/instances/` — 105 schema-valid BIM* instances (campaign input).
- `out/bimstar-priced/reports/` — full provenance: `pricing_trace.csv`, `latency_trace.csv`,
  `security_trace.csv`, `budget_trace.csv`, `candidate_generation_summary.csv`, validation report.
- `out/results/corpus_summary.csv` — derived instance attributes.
- `out/results/figures/preprocessing_*.pdf|png` — figures for the paper.

Next: `02_campaign_execution.ipynb`.